# ChEMBL — Bioactive Molecules and Drug Discovery Data

**ChEMBL** is a manually curated database of bioactive molecules with drug-like properties, maintained by the European Bioinformatics Institute (EBI). It integrates binding, functional, and ADMET (absorption, distribution, metabolism, excretion, and toxicity) data extracted from the primary scientific literature, together with approved drug information and clinical development candidates.

Key data types:
| Data type | Description |
|---|---|
| **Molecules** | Small molecules with physicochemical properties, SMILES, InChI keys |
| **Bioactivities** | Assay results (IC50, Ki, EC50, etc.) linking molecules to targets |
| **Targets** | Proteins, protein complexes, cell lines, and organisms |
| **Assays** | Experimental protocols linking bioactivities to targets |
| **Drugs** | Approved drugs (max_phase = 4) with indication and approval year |
| **Mechanisms** | Drug mechanisms of action and target bindings |

**Reference:** Mendez et al. (2019), *Nucleic Acids Research*, ChEMBL: towards direct deposition of bioassay data

**API base:** `https://www.ebi.ac.uk/chembl/api/data/`

# TODO

* [x] **Ingest data**
    * [x] Connect to ChEMBL REST API and confirm access (fetch ibuprofen, print key fields)
    * [x] Fetch all approved drugs (max_phase = 4) with full pagination
    * [x] Parse approved drugs into a Polars DataFrame with correct dtypes
    * [x] Fetch IC50 bioactivity data for EGFR kinase (CHEMBL205) with full pagination
    * [x] Parse bioactivities into a Polars DataFrame
* [ ] **Explore and clean**
    * [ ] Summarise approved drug physicochemical property distributions (MW, AlogP, HBD/HBA)
    * [ ] Examine Ro5 (Lipinski's Rule of Five) compliance across the approved drug set
    * [ ] Inspect pChEMBL value distributions for EGFR activities; handle missing values
    * [ ] Filter bioactivities by standard_type and data quality flags
* [ ] **Analysis**
    * [ ] Compare property profiles of approved drugs vs. clinical candidates
    * [ ] Identify EGFR inhibitor series by clustering on pChEMBL values
    * [ ] Correlate structural features (AlogP, MW) with EGFR potency
    * [ ] Explore first approval year trends over time
* [ ] **Visualization**
    * [ ] Property distribution plots for approved drugs (MW, AlogP, HBD/HBA histograms)
    * [ ] pChEMBL activity landscape for EGFR inhibitors
    * [ ] Scatter plot: MW vs. AlogP coloured by Ro5 violations
    * [ ] Timeline of drug approvals by decade
* [ ] **Statistical analysis**
    * [ ] Discuss error propagation in IC50 → pChEMBL conversion (−log₁₀ transform)
    * [ ] Multiple assay aggregation: when to take median vs. mean across replicates
    * [ ] Discuss activity cliff detection and its statistical basis

In [ ]:
import requests
import time
import json
from pathlib import Path

import polars as pl

## 1. Ingest Data

### 1.1 Connect to ChEMBL API and Confirm Access

In [ ]:
CHEMBL_BASE = "https://www.ebi.ac.uk/chembl/api/data"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def chembl_get(endpoint: str, params: dict = None) -> dict:
    """
    Send a GET request to the ChEMBL REST API.

    Parameters
    ----------
    endpoint : str
        API path relative to CHEMBL_BASE (e.g. "molecule/CHEMBL521").
    params : dict, optional
        Query parameters; ``format=json`` is added automatically.

    Returns
    -------
    dict
        Parsed JSON response.
    """
    url = f"{CHEMBL_BASE}/{endpoint}"
    p = {"format": "json"}
    p.update(params or {})
    resp = requests.get(url, params=p, timeout=30)
    resp.raise_for_status()
    return resp.json()

# Connectivity check: fetch ibuprofen (CHEMBL521) and print key fields
ibu = chembl_get("molecule/CHEMBL521")
props = ibu.get("molecule_properties", {}) or {}

print(f"Name            : {ibu.get('pref_name')}")
print(f"Formula         : {props.get('full_molformula')}")
print(f"Max phase       : {ibu.get('max_phase')}")
print(f"Molecular weight: {props.get('full_mwt')}")

### 1.2 Fetch All Approved Drugs (max_phase = 4)

In [ ]:
DRUGS_CACHE = DATA_DIR / "chembl_approved_drugs.json"
PAGE_SIZE = 100   # maximum records per page the ChEMBL API will return

def fetch_approved_drugs(cache_path: Path = DRUGS_CACHE) -> list[dict]:
    """
    Fetch all ChEMBL molecules with max_phase = 4 (approved drugs) via the
    paginated /molecule endpoint. Results are cached to disk so subsequent
    runs skip the download.

    Parameters
    ----------
    cache_path : Path
        Where to write/read the cached JSON list.

    Returns
    -------
    list[dict]
        One dict per approved drug molecule.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    all_drugs = []
    offset = 0
    while True:
        resp = chembl_get(
            "molecule",
            {"max_phase": 4, "limit": PAGE_SIZE, "offset": offset},
        )
        batch = resp.get("molecules", [])
        if not batch:
            break
        all_drugs.extend(batch)
        total = resp.get("page_meta", {}).get("total_count", "?")
        print(f"  Fetched {len(all_drugs):>5} / {total}", end="\r")
        offset += PAGE_SIZE
        time.sleep(0.3)   # polite delay between paginated calls

    print(f"\nDone. Fetched {len(all_drugs)} approved drugs.")
    cache_path.write_text(json.dumps(all_drugs))
    return all_drugs

drugs_raw = fetch_approved_drugs()
print(f"Total records: {len(drugs_raw)}")

### 1.3 Parse Approved Drugs into a Polars DataFrame

In [ ]:
def flatten_drug(m: dict) -> dict:
    """
    Flatten a single raw ChEMBL molecule record into a row-friendly dict,
    pulling physicochemical properties out of the nested
    ``molecule_properties`` sub-object.

    Parameters
    ----------
    m : dict
        Raw molecule record from the ChEMBL API.

    Returns
    -------
    dict
        Flat dict suitable for a single DataFrame row.
    """
    props = m.get("molecule_properties") or {}   # may be None for mixtures/biologics
    return {
        "chembl_id":           m.get("molecule_chembl_id"),
        "pref_name":           m.get("pref_name"),
        "max_phase":           m.get("max_phase"),
        "molecular_formula":   props.get("full_molformula"),
        "molecular_weight":    props.get("full_mwt"),
        "alogp":               props.get("alogp"),
        "hbd_count":           props.get("hbd"),       # hydrogen-bond donors
        "hba_count":           props.get("hba"),       # hydrogen-bond acceptors
        "ro5_violations":      props.get("num_ro5_violations"),   # Lipinski Ro5
        "indication_class":    m.get("indication_class"),
        "first_approval_year": m.get("first_approval"),
    }

rows = [flatten_drug(m) for m in drugs_raw]

drugs = pl.DataFrame(rows).with_columns([
    pl.col("max_phase").cast(pl.Int8, strict=False),
    pl.col("molecular_weight").cast(pl.Float64, strict=False),
    pl.col("alogp").cast(pl.Float64, strict=False),
    pl.col("hbd_count").cast(pl.Int16, strict=False),
    pl.col("hba_count").cast(pl.Int16, strict=False),
    pl.col("ro5_violations").cast(pl.Int8, strict=False),
    pl.col("first_approval_year").cast(pl.Int16, strict=False),
])

print(f"Shape : {drugs.shape}")
print(f"\nDtypes:")
print(drugs.schema)
print()
drugs.head(5)

### 1.4 Fetch EGFR Kinase IC50 Bioactivity Data

In [ ]:
EGFR_TARGET = "CHEMBL205"   # Epidermal growth factor receptor (EGFR) kinase
ACTIVITIES_CACHE = DATA_DIR / "chembl_egfr_activities.json"

def fetch_target_activities(
    target_chembl_id: str,
    standard_type: str = "IC50",
    cache_path: Path = ACTIVITIES_CACHE,
) -> list[dict]:
    """
    Fetch all bioactivity measurements of a given standard_type for a target
    from ChEMBL, paginating through the /activity endpoint.

    Parameters
    ----------
    target_chembl_id : str
        ChEMBL ID of the target, e.g. ``"CHEMBL205"`` for EGFR.
    standard_type : str
        Assay measurement type to filter on, e.g. ``"IC50"`` or ``"Ki"``.
    cache_path : Path
        Where to write/read the cached JSON list.

    Returns
    -------
    list[dict]
        One dict per activity record.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    all_activities = []
    offset = 0
    while True:
        resp = chembl_get(
            "activity",
            {
                "target_chembl_id": target_chembl_id,
                "standard_type":    standard_type,
                "limit":            PAGE_SIZE,
                "offset":           offset,
            },
        )
        batch = resp.get("activities", [])
        if not batch:
            break
        all_activities.extend(batch)
        total = resp.get("page_meta", {}).get("total_count", "?")
        print(f"  Fetched {len(all_activities):>6} / {total}", end="\r")
        offset += PAGE_SIZE
        time.sleep(0.3)   # polite delay

    print(f"\nDone. Fetched {len(all_activities)} activity records.")
    cache_path.write_text(json.dumps(all_activities))
    return all_activities

egfr_raw = fetch_target_activities(EGFR_TARGET)
print(f"Total activity records: {len(egfr_raw)}")

### 1.5 Parse Bioactivities into a Polars DataFrame

In [ ]:
def flatten_activity(a: dict) -> dict:
    """
    Flatten a single ChEMBL activity record into a row-friendly dict.

    Parameters
    ----------
    a : dict
        Raw activity record from the ChEMBL /activity endpoint.

    Returns
    -------
    dict
        Flat dict with the key fields needed for downstream analysis.
    """
    return {
        "molecule_chembl_id": a.get("molecule_chembl_id"),
        "standard_type":      a.get("standard_type"),      # e.g. IC50
        "standard_value":     a.get("standard_value"),     # numeric value
        "standard_units":     a.get("standard_units"),     # e.g. nM
        "pchembl_value":      a.get("pchembl_value"),      # -log10(standard_value in M)
        "assay_chembl_id":    a.get("assay_chembl_id"),
    }

activity_rows = [flatten_activity(a) for a in egfr_raw]

activities = pl.DataFrame(activity_rows).with_columns([
    pl.col("standard_value").cast(pl.Float64, strict=False),
    pl.col("pchembl_value").cast(pl.Float64, strict=False),
])

# ── Print summaries for both DataFrames ──────────────────────────────────────
print("=== Approved drugs DataFrame ===")
print(f"Shape : {drugs.shape}")
print(f"Dtypes: {drugs.dtypes}")
print()
print(drugs.head(3))

print("\n=== EGFR IC50 activities DataFrame ===")
print(f"Shape : {activities.shape}")
print(f"Dtypes: {activities.dtypes}")
print()
print(activities.head(3))